In [20]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge

In [3]:
df = pd.read_excel("воронка.xlsx")

In [4]:
df = df.rename(columns={
    "Viewed": "viewed",
    "Add-to-Cart": "add_to_cart",
    "Orders": "orders",
    "Add-to-Cart Conversion (%)": "add_to_cart_conversion",
    "Order Conversion (%)": "order_conversion",
    "Add-to-Favorites": "add_to_favorites",
    "актуальные охваты": "reach",
    "продукт": "product",
    "CPM": "cpm",
    "затраты": "cost",
    "формат": "format",
    "месяц": "month",
    "платформа": "platform"
})

In [5]:
text_cols = ["product", "format", "month", "platform"]
for col in text_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({"nan": np.nan, "none": np.nan, "": np.nan, "-": np.nan})
    )

In [6]:
num_cols = [
    "viewed",
    "add_to_cart",
    "orders",
    "add_to_cart_conversion",
    "order_conversion",
    "add_to_favorites",
    "reach",
    "cpm",
    "cost"
]

for col in num_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [7]:
group_cols = ["platform", "format", "month"]
fill_numeric_cols = [
    "viewed",
    "add_to_cart",
    "add_to_favorites",
    "reach",
    "cpm",
    "cost",
    "add_to_cart_conversion"
]
for col in fill_numeric_cols:
    df[col] = df[col].fillna(df.groupby(group_cols)[col].transform("median"))
    df[col] = df[col].fillna(df[col].median())

In [8]:
product_freq = df["product"].value_counts(normalize=True)
df["product_freq"] = df["product"].map(product_freq)
df["product_freq"] = df["product_freq"].fillna(df["product_freq"].median())

In [9]:
for col in ["format", "month", "platform"]:
    df[col] = df[col].fillna("unknown")

In [10]:
df = df[df["orders"].notna()].copy()

In [11]:
feature_cols = [
    "viewed",
    "add_to_cart",
    "add_to_favorites",
    "reach",
    "cpm",
    "cost",
    "product_freq",
    "format",
    "month",
    "platform"
]

X = df[feature_cols]
y = df["orders"]

In [14]:
numeric_features = ["viewed", "add_to_cart", "add_to_favorites", "reach", "cpm", "cost", "product_freq"]
categorical_features = ["format", "month", "platform"]
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [21]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=1.0))
])

model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

In [31]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2:   {r2:.4f}")

MAE:  7.1708
RMSE: 162.7609
R2:   0.2143


In [32]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefs = pd.DataFrame({
    "feature": feature_names,
    "coef": model.named_steps["regressor"].coef_
}).sort_values(by="coef", key=np.abs, ascending=False)

print(coefs.head(20))

                    feature       coef
0               num__viewed  25.198511
1          num__add_to_cart  15.776562
11        cat__format_обзор   3.391721
8          cat__format_влог  -3.174668
5                 num__cost   2.709304
17       cat__month_февраль   2.673923
2     num__add_to_favorites  -2.112460
20      cat__platform_инста  -2.043691
21         cat__platform_тг   1.730604
9   cat__format_грвм/рутина  -1.677035
15       cat__month_декабрь  -1.565851
13        cat__format_совет  -1.465060
12          cat__format_пов   1.158758
23       cat__platform_ютуб  -1.128819
6         num__product_freq   1.045523
18        cat__month_январь  -1.037946
19         cat__platform_вк   0.736332
7       cat__format_unknown   0.720422
22     cat__platform_тикток   0.705574
10   cat__format_интеграция   0.549204


In [30]:
df.head()

,viewed,add_to_cart,orders,add_to_cart_conversion,order_conversion,add_to_favorites,reach,product,cpm,cost,format,month,platform,product_freq
0,47.0,4.0,48.0,22.0,NaN,2.0,482800,tsh213,9500.0,95000.0,грвм/рутина,март,тикток,0.027223
1,7.0,0.0,0.0,0.0,0.0,3.0,41500,tsh90,6000.0,60000.0,тутор,март,тикток,0.119782
2,26.0,3.0,1.0,12.0,33.0,2.0,41500,tsh90,6000.0,60000.0,тутор,февраль,тикток,0.119782
3,47.0,5.0,4.0,11.0,80.0,6.0,41500,tsh90,6000.0,60000.0,тутор,январь,тикток,0.119782
4,574.0,4.0,12.0,14.0,NaN,2.0,17000,tsh213,5512.0,93700.0,обзор,март,тг,0.027223
